In [1]:
import os
import time
import datetime
import pandas as pd

# Importaciones de Alpaca
from alpaca.trading.client import TradingClient
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit # <-- TimeFrameUnit añadido
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

# Sustituye con tus credenciales de Paper Trading
API_KEY = "PK2CNGKUVBXK5XM75N64PIQIRL"
SECRET_KEY = "G1Ag2nzs6cFosZvBRp1QKTFLvw5nYBvNidn1W8Em9zNE"

trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = CryptoHistoricalDataClient()

## 2. Obtención de Datos y Lógica de la Estrategia
Esta función descarga los últimos precios y calcula si debemos comprar o vender.

In [2]:
def get_signal(symbol="BTC/USD"):
    try:
        ahora_utc = datetime.datetime.now(datetime.timezone.utc)
        # Pedimos 1500 minutos hacia atrás (100 velas de 15 min aprox)
        start_time = ahora_utc - datetime.timedelta(minutes=1500)
        
        request_params = CryptoBarsRequest(
            symbol_or_symbols=[symbol],
            timeframe=TimeFrame(15, TimeFrameUnit.Minute), # Intervalo de 15 minutos
            start=start_time,
            end=ahora_utc
        )
        
        bars = data_client.get_crypto_bars(request_params).df
        
        if bars.empty:
            print("Esperando datos del mercado...")
            return "wait"
            
        df = bars.loc[symbol].copy()
        
        # --- LÓGICA DE VELAS CERRADAS Y CÁLCULO DE DELAY ---
        ultima_vela_ts = df.index[-1]
        retraso_total = ahora_utc - ultima_vela_ts
        
        # Si la última vela tiene menos de 15 minutos de antigüedad, significa que sigue abierta.
        # La eliminamos del cálculo para no "repintar" indicadores.
        if retraso_total.total_seconds() < 900: 
            df = df.iloc[:-1] # Quitamos la última fila
            ultima_vela_ts = df.index[-1] # Actualizamos el timestamp a la última vela cerrada
        
        # Calculamos el delay exacto desde que se cerró la última vela válida
        delay_segundos = (ahora_utc - ultima_vela_ts).total_seconds()
        mins = int(delay_segundos // 60)
        segs = int(delay_segundos % 60)
        
        print(f"⏱️  Delay API / Última vela: {mins} minutos y {segs} segundos de antigüedad.")
        
        # --- CÁLCULO DE INDICADORES ---
        df['ema_9'] = df['close'].ewm(span=9, adjust=False).mean()
        df['ema_21'] = df['close'].ewm(span=21, adjust=False).mean()
        
        delta = df['close'].diff()
        gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
        loss = -delta.clip(upper=0).ewm(alpha=1/14, adjust=False).mean()
        rs = gain / loss
        df['rsi_14'] = 100 - (100 / (1 + rs))
        
        # --- LECTURA DE DATOS ---
        current = df.iloc[-1]   # Última vela cerrada
        previous = df.iloc[-2]  # Penúltima vela cerrada
        
        hora_actual = datetime.datetime.now().strftime('%H:%M:%S')
        print(f"[{hora_actual}] BTC: ${current['close']:.2f} | EMA9: ${current['ema_9']:.2f} | EMA21: ${current['ema_21']:.2f} | RSI: {current['rsi_14']:.1f}")
        
        # --- CONDICIONES DE LA ESTRATEGIA ---
        cruce_alcista = (current['ema_9'] > current['ema_21']) and (previous['ema_9'] <= previous['ema_21'])
        rsi_saludable = 50 < current['rsi_14'] < 70
        cruce_bajista = current['ema_9'] < current['ema_21']
        
        if cruce_alcista and rsi_saludable: return "buy"
        elif cruce_bajista: return "sell"
        return "wait"
            
    except Exception as e:
        print(f"Error calculando estrategia: {e}")
        return "wait"

## 3. Ejecución de Operaciones
Alpaca permite trading fraccional en cripto, lo cual es ideal para tu presupuesto de 1$.

In [ ]:
def execute_trade(side, symbol="BTC/USD"):
    pos_symbol = symbol.replace("/", "") 
    
    positions = trading_client.get_all_positions()
    btc_position = next((p for p in positions if p.symbol == pos_symbol), None)

    # REVISIÓN DE STOP LOSS Y TAKE PROFIT (Ajustado para 15 minutos)
    if btc_position:
        profit_pct = float(btc_position.unrealized_plpc)
        print(f"📊 ESTADO: Posición abierta en {symbol} | PnL actual: {profit_pct*100:.3f}%")
        
        if profit_pct >= 0.015: # +1.5% Take Profit
            trading_client.close_all_positions()
            print(f"✅ TAKE PROFIT: Posición cerrada con ganancia del {profit_pct*100:.2f}%")
            return 
            
        elif profit_pct <= -0.01: # -1.0% Stop Loss
            trading_client.close_all_positions()
            print(f"🛑 STOP LOSS: Posición cerrada con pérdida del {profit_pct*100:.2f}%")
            return 
    else:
        print(f"👀 ESTADO: No hay posiciones abiertas en {symbol}.")

    # EJECUCIÓN DE SEÑALES
    if side == "buy":
        if not btc_position:
            order_data = MarketOrderRequest(
                symbol=symbol,
                notional=100.00,
                side=OrderSide.BUY,
                time_in_force=TimeInForce.GTC
            )
            trading_client.submit_order(order_data)
            print("🚀 ACCIÓN: Orden de COMPRA enviada ($10). Estrategia alcista confirmada en 15m.")
        else:
            print("⏳ IGNORADO: Señal de COMPRA, pero YA TIENES posición. Esperando...")

    elif side == "sell":
        if btc_position:
            trading_client.close_all_positions()
            print("🔄 ACCIÓN: Posición CERRADA (Venta). El precio cruzó la media hacia abajo.")
        else:
            print("⏳ IGNORADO: Señal de VENTA, pero NO HAY posición. Esperando compra...")
            
    elif side == "wait":
        if btc_position:
            print("⏸️  ACCIÓN: Ninguna. Manteniendo posición abierta...")
        else:
            print("⏸️  ACCIÓN: Ninguna. Esperando cruce de medias...")

## 4. El Bucle Principal (El Bot en marcha)
Este bloque mantiene al bot escuchando el mercado cada minuto.

In [ ]:
print("--- Bot Iniciado en modo Paper Trading ---")

while True:
    print("-" * 60)
    signal = get_signal()
    execute_trade(signal)
            
    # Revisamos el mercado cada 60 segundos. 
    # Aunque la vela sea de 15m, revisar cada minuto nos permite ejecutar
    # el Stop Loss o Take Profit inmediatamente si el precio se dispara.
    time.sleep(60)

--- Bot Iniciado en modo Paper Trading ---
------------------------------------------------------------
⏱️  Delay API / Última vela: 24 minutos y 22 segundos de antigüedad.
[12:54:23] BTC: $70258.99 | EMA9: $69867.37 | EMA21: $69585.61 | RSI: 80.5
👀 ESTADO: No hay posiciones abiertas en BTC/USD.
⏸️  ACCIÓN: Ninguna. Esperando cruce de medias...
------------------------------------------------------------
⏱️  Delay API / Última vela: 25 minutos y 23 segundos de antigüedad.
[12:55:23] BTC: $70258.99 | EMA9: $69867.37 | EMA21: $69585.61 | RSI: 80.5
👀 ESTADO: No hay posiciones abiertas en BTC/USD.
⏸️  ACCIÓN: Ninguna. Esperando cruce de medias...
------------------------------------------------------------
⏱️  Delay API / Última vela: 26 minutos y 23 segundos de antigüedad.
[12:56:23] BTC: $70258.99 | EMA9: $69867.37 | EMA21: $69585.61 | RSI: 80.5
👀 ESTADO: No hay posiciones abiertas en BTC/USD.
⏸️  ACCIÓN: Ninguna. Esperando cruce de medias...
---------------------------------------------